# API нейрошлюза `ai.rt.ru`

Справочник рабочих вызовов из этого ноутбука: функция + пример обращения.
Общий префикс: `https://ai.rt.ru/api/1.0`.

| Сервис | Метод | URL |
|---|---|---|
| Авторизация | POST | `/auth/authentication` |
| Claude | POST | `/claude/chat` |
| Леопольд / Qwen | POST | `/llama/chat` |
| Gemini | POST | `/gemini/chat` |
| ChatGPT | POST | `/chatgpt/chat` |
| Perplexity | POST | `/perplexity/chat` |
| ИИ-агент | POST | `/aiAgent/chat` |
| Умный поиск: коллекции | GET | `/searchByFile/collections` |
| Умный поиск: вопрос | POST | `/searchByFile/search` |
| Умный поиск: файлы коллекции | GET | `/searchByFile/collectionsFiles` |
| Умный поиск: загрузка файла | POST | `/searchByFile/uploadFile` |
| Умный поиск: удаление файла | DELETE | `/searchByFile/deleteFile` |
| OCR Qwen Omni | POST multipart | `/llama/chatMulti` |
| OCR Llama 3.2 Vision | POST multipart | `/llama3_2/chat` |


## 0. Общие настройки


In [2]:
import json
import os
import uuid
from io import BytesIO
from pathlib import Path

import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE = "https://ai.rt.ru/api/1.0"
TIMEOUT = 300

TOKEN = "eyJhbGciOiJIUzM4NCJ9.eyJzY29wZXMiOlsiYWRtaW5fZWRpdF9ib3QiLCJhZG1pbl9yZWdpc3RyYXRpb25fdXNlciIsImFkbWluX3JlcG9ydHNfc3RhdGlzdGljIiwiYWlBZ2VudCIsIm1lc3NlbmdlciIsImJpIiwicGx1Z2luIiwiYW5ub3RhdG9yIiwiY2hhdEdwdCIsImNsYXVkZSIsImNvbXBhbnlfdDJfYXBpIiwiZGVlcFNlZWsiLCJmbHV4IiwiZ2VtaW5pIiwiZ2lnYUNoYXQiLCJncHRBdWRpbyIsImdwdEltZyIsImdwdFNwZWVjaCIsImdyb2siLCJrYW5kaW5za3kiLCJsbGFtYSIsImxsYW1hMyIsImxsYW1hMy4yIiwibWlkam91cm5leSIsIm1pc3RyYWwiLCJuYW5vQmFuYW5hIiwib2NyIiwicGVycGxleGl0eSIsInByZXNlbnRhdGlvbiIsInNkIiwic2VhcmNoQnlGaWxlIiwic2VhcmNoQnlGaWxlUHJpdmF0ZSIsInNlYXJjaEJ5VGFibGUiLCJzcGVlY2hFbGV2ZW4iLCJzcGVlY2hMb2NhbCIsInN1bW1hcnkiLCJ0cmFuc2xhdGVEZWVwbCIsInRyYW5zbGF0ZUZpbGUiLCJ0cmFuc2xhdGVMb2NhbCIsInZhc2lsaXNhIiwid2hpc3BlciIsInlhQXJ0IiwieWFDaGF0IiwieWFTdW1tYXJ5TGluayJdLCJzdWIiOiJ0dXpfdmFzaWxpeS52b3JvYmV2IiwiaWF0IjoxNzg2NjE3MTU0LCJleHAiOjE4MTgxNTMxNTR9.iJEwUBxW4jr01-ePGIeqKFHuWOSK-6yf141B2QvSI8egosXbNzISUN6TpkeFfPgc"


def headers(token=TOKEN, json_content=True):
    h = {"Authorization": f"Bearer {token}"}
    if json_content:
        h["Content-Type"] = "application/json"
    else:
        h["accept"] = "*/*"
    return h


def extract_text(js):
    """Достаёт текст ответа из типичных форматов нейрошлюза."""
    if isinstance(js, str):
        return js
    if isinstance(js, dict) and js.get("error"):
        return f"Error {js.get('status_code')}: {js.get('text')}"
    if isinstance(js, dict):
        message = js.get("message")
        if isinstance(message, dict) and message.get("content"):
            return str(message["content"]).strip()
        choices = js.get("choices")
        if isinstance(choices, list) and choices:
            msg = choices[0].get("message", {})
            return str(msg.get("content", "")).strip()
        return str(js)
    if isinstance(js, list) and js:
        first = js[0]
        if isinstance(first, dict):
            message = first.get("message")
            if isinstance(message, dict) and "content" in message:
                return str(message["content"]).strip()
            if isinstance(message, str):
                return message.strip()
        return str(first)
    return str(js)


def post_json(url, payload, token=TOKEN, timeout=TIMEOUT):
    r = requests.post(url, headers=headers(token), json=payload, verify=False, timeout=timeout)
    if r.status_code != 200:
        return f"Error {r.status_code}: {r.text}"
    return r.json()


## 1. Авторизация

Если токен выше ещё живой, эту ячейку можно не запускать.


In [3]:
def authenticate(login: str, password: str, session: bool = True) -> str:
    url = f"{BASE}/auth/authentication"
    payload = {"login": login, "password": password, "session": session}
    r = requests.post(
        url,
        json=payload,
        headers={"Content-Type": "application/json"},
        verify=False,
        timeout=60,
    )
    if r.status_code != 200:
        raise RuntimeError(f"Ошибка авторизации {r.status_code}: {r.text}")
    token = r.json().get("token")
    if not token:
        raise RuntimeError(f"Токен не пришёл: {r.text}")
    return token


# token = authenticate("login@rt.ru", "password")
print("Используется TOKEN длиной", len(TOKEN), "символов")


Используется TOKEN длиной 906 символов


## 2. Claude

`POST /claude/chat`  
Модели из рабочих вызовов: `claude-haiku-4-5`, `claude-opus-4-6`.


In [ ]:
def claude_chat(
    prompt: str,
    model: str = "claude-haiku-4-5",
    temperature: float = 1.0,
    max_tokens: int = 7000,
    chat_uuid: str = "00000000-0000-0000-0000-000000000000",
    top_p=None,
    top_k=None,
    tools=None,
    token: str = TOKEN,
):
    payload = {
        "uuid": chat_uuid,
        "chat": {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "top_p": top_p,
            "top_k": top_k,
            "maxTokens": max_tokens,
            "temperature": temperature,
            "tools": tools,
        },
    }
    return post_json(f"{BASE}/claude/chat", payload, token=token)


answer = claude_chat("Привет, как дела?", model="claude-opus-4-6", temperature=1, max_tokens=7000)
print(extract_text(answer))


## 3. Леопольд / Qwen

`POST /llama/chat`

Два рабочих формата тела:
- `contents` — как у Qwen;
- `user_message` + `chat_history` — как у Леопольда.

Модели из примеров:
- `Qwen/Qwen2.5-72B-Instruct`
- `Qwen/Qwen3-Next-80B-A3B-Instruct-FP8`


In [6]:
def llama_chat(
    prompt: str,
    model: str = "Qwen/Qwen2.5-72B-Instruct",
    temperature: float = 0.5,
    max_new_tokens: int = 1536,
    system_prompt: str = "Ты — Сайга, русскоязычный автоматический ассистент. Ты разговариваешь с людьми и помогаешь им.",
    chat_history=None,
    chat_uuid=None,
    token: str = TOKEN,
):
    chat = {
        "model": model,
        "system_prompt": system_prompt,
        "max_new_tokens": max_new_tokens,
        "no_repeat_ngram_size": 15,
        "repetition_penalty": 1.1,
        "temperature": temperature,
        "top_k": 40,
        "top_p": 0.9,
    }
    if chat_history:
        chat["user_message"] = prompt
        chat["chat_history"] = chat_history
        chat["message_template"] = "<s>{role}\n{content}</s>"
        chat["response_template"] = "<s>bot\n"
    else:
        chat["contents"] = [{"type": "text", "text": prompt}]

    payload = {"chat": chat}
    if chat_uuid:
        payload["uuid"] = chat_uuid
    return post_json(f"{BASE}/lleopold/chatMulti", payload, token=token)


print(extract_text(llama_chat("привет", temperature=0.5, max_new_tokens=220)))

print(
    extract_text(
        llama_chat(
            "привет",
            model="Qwen/Qwen3-Next-80B-A3B-Instruct-FP8",
            temperature=0.5,
            max_new_tokens=220,
        )
    )
)


Привет! Я Ллеопольд, экспертный ИИ-ассистент компании Ростелеком. Как я могу вам помочь сегодня?
Привет! 😊  
Я — Ллеопольд, ваш ассистент. Чем могу помочь сегодня?


In [18]:
history_uuid = "019b4b58-ade3-71d6-9fe6-250f4e8ca91c"
history = [{"role": "user", "content": "Моего кота зовут феликс"}]
print(
    extract_text(
        llama_chat(
            "как зовут моего кота?",
            chat_history=history,
            chat_uuid=history_uuid,
        )
    )
)


Error 500: {"title":"Внутренняя ошибка","message":"Что-то пошло не так","errorCode":"INTERNAL_ERROR","requestId":"01a01589-5e33-7668-b4bd-ac5bdad5a836","additionalData":null}


## 4. Gemini

`POST /gemini/chat`  
Модель из примера: `google/gemini-2.5-flash`.


In [ ]:
def gemini_chat(
    prompt: str,
    model: str = "google/gemini-2.5-flash",
    temperature: float = 0.0,
    max_output_tokens: int = 4500,
    internet_search: bool = False,
    chat_uuid: str = "00000000-0000-0000-0000-000000000001",
    token: str = TOKEN,
):
    payload = {
        "uuid": chat_uuid,
        "chat": {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "generationConfig": {
                "temperature": temperature,
                "maxOutputTokens": max_output_tokens,
                "topP": 0.2,
                "topK": 4,
            },
            "internetSearch": internet_search,
        },
    }
    return post_json(f"{BASE}/gemini/chat", payload, token=token)


print(extract_text(gemini_chat("привет, как дела?")))


## 5. ChatGPT

`POST /chatgpt/chat`  
Модели из примеров: `gpt-4.1-nano`, `gpt-5.2`.


In [ ]:
def chatgpt_chat(
    prompt: str,
    model: str = "gpt-4.1-nano",
    temperature: float = 0.0,
    max_completion_tokens: int = 4500,
    top_p: float = 0.2,
    chat_uuid: str = "00000000-0000-0000-0000-000000000001",
    token: str = TOKEN,
):
    payload = {
        "uuid": chat_uuid,
        "chat": {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "top_p": top_p,
            "n": 1,
            "max_completion_tokens": max_completion_tokens,
            "presence_penalty": 0,
            "frequency_penalty": 0,
            "parallel_tool_calls": True,
        },
    }
    return post_json(f"{BASE}/chatgpt/chat", payload, token=token)


print(extract_text(chatgpt_chat("привет, как дела?", model="gpt-5.2")))


## 6. Perplexity

`POST /perplexity/chat`  
Модель из примера: `sonar`. Можно ограничить поиск доменом через `search_domain_filter`.


In [ ]:
def perplexity_chat(
    prompt: str,
    model: str = "sonar",
    temperature: float = 0.5,
    max_tokens: int = 512,
    search_mode: str = "web",
    search_domain_filter=None,
    token: str = TOKEN,
):
    chat = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "search_mode": search_mode,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": 0.9,
        "stream": False,
        "web_search_options": {"search_context_size": "high"},
    }
    if search_domain_filter:
        chat["search_domain_filter"] = search_domain_filter
    return post_json(f"{BASE}/perplexity/chat", {"chat": chat}, token=token)


raw = perplexity_chat(
    "найди информацию по проекту чистая вода",
    search_domain_filter=["https://www.fedstat.ru/"],
)
print(extract_text(raw))


## 7. ИИ-агент

`POST /aiAgent/chat`  
`agent_id` берётся из карточки агента в нейрошлюзе.


In [ ]:
def agent_chat(
    user_message: str,
    agent_id: str,
    chat_uuid: str | None = None,
    token: str = TOKEN,
):
    if chat_uuid is None:
        chat_uuid = str(uuid.uuid4())
    payload = {
        "uuid": chat_uuid,
        "chat": {
            "messages": [{"role": "user", "content": user_message}],
            "agent_id": agent_id,
        },
    }
    return post_json(f"{BASE}/aiAgent/chat", payload, token=token)


AGENT_ID = "019e5b45-e403-7ec0-9109-ac6741dd6098"
print(extract_text(agent_chat("Что такое ВХР", AGENT_ID)))


## 8. Умный поиск по файлам

Эндпоинты:
- `GET /searchByFile/collections` — список коллекций;
- `POST /searchByFile/search` — вопрос к коллекции;
- `GET /searchByFile/collectionsFiles?code=...` — файлы коллекции;
- `POST /searchByFile/uploadFile` — загрузка файла;
- `DELETE /searchByFile/deleteFile?code=...` — удаление файла.


In [ ]:
def get_collections(token: str = TOKEN):
    r = requests.get(
        f"{BASE}/searchByFile/collections",
        headers=headers(token),
        verify=False,
        timeout=60,
    )
    if r.status_code != 200:
        return f"Error {r.status_code}: {r.text}"
    return r.json()


def search_by_file(
    collection_code: str,
    query: str,
    llm_name: str,
    temperature,
    search_limit,
    system_prompt: str,
    token: str = TOKEN,
):
    payload = {
        "searchByFileMessageRequest": {
            "collectionCode": collection_code,
            "query": query,
            "llmName": llm_name,
            "temperature": temperature,
            "searchLimit": search_limit,
            "systemPrompt": system_prompt,
        }
    }
    return post_json(f"{BASE}/searchByFile/search", payload, token=token)


def get_collection_files(collection_code: str, page_size: int = 1000, token: str = TOKEN):
    r = requests.get(
        f"{BASE}/searchByFile/collectionsFiles",
        headers=headers(token, json_content=False),
        params={"code": collection_code, "pageSize": page_size},
        verify=False,
        timeout=60,
    )
    r.raise_for_status()
    data = r.json()
    if isinstance(data, dict) and isinstance(data.get("list"), list):
        return data["list"]
    if isinstance(data, list):
        return data
    return []


def upload_file(collection_code: str, file_path: str, token: str = TOKEN):
    with open(file_path, "rb") as f:
        r = requests.post(
            f"{BASE}/searchByFile/uploadFile",
            headers={"Authorization": f"Bearer {token}"},
            data={"collectionCode": collection_code},
            files={"file": f},
            verify=False,
            timeout=TIMEOUT,
        )
    if r.status_code != 200:
        return f"Error {r.status_code}: {r.text}"
    try:
        return r.json()
    except Exception:
        return r.text


def delete_file(file_code: str, token: str = TOKEN):
    r = requests.delete(
        f"{BASE}/searchByFile/deleteFile",
        headers={"Authorization": f"Bearer {token}"},
        params={"code": file_code},
        verify=False,
        timeout=60,
    )
    if r.status_code != 200:
        return f"Error {r.status_code}: {r.text}"
    return r.text or "ok"


In [ ]:
collections = get_collections()
if isinstance(collections, str):
    print(collections)
else:
    print(f"Коллекций: {len(collections)}")
    for c in collections[:5]:
        print(f"- {c.get('name')}  code={c.get('code')}  llm={c.get('llmName')}")

COLLECTION_NAME = "BUH_template"
coll = None
if isinstance(collections, list):
    coll = next((obj for obj in collections if obj.get("name") == COLLECTION_NAME), None)

if coll:
    raw = search_by_file(
        collection_code=coll["code"],
        query="Какой документ подходит для акта сверки?",
        llm_name=coll["llmName"],
        temperature=coll["temperature"],
        search_limit=coll["searchLimit"],
        system_prompt=coll["systemPrompt"],
    )
    print(extract_text(raw))
else:
    print(f"Коллекция {COLLECTION_NAME!r} не найдена, поиск пропущен")


In [ ]:
# Файлы коллекции / загрузка / удаление.
# Раскомментируйте и подставьте свои пути.

# COLLECTION_CODE = "01982d55-fc28-782e-8b81-406a20580d11"
# FILE_PATH = "/path/to/file.docx"

# files_json = get_collection_files(COLLECTION_CODE)
# print("Файлы до загрузки:")
# for f in files_json:
#     print(f.get("fileName"), "→", f.get("code"))

# upload_file(COLLECTION_CODE, FILE_PATH)
# files_json = get_collection_files(COLLECTION_CODE)
# print("Файлы после загрузки:")
# for f in files_json:
#     print(f.get("fileName"), "→", f.get("code"))

# удаление по коду файла, не коллекции
# print(delete_file("0198e1e9-6d89-727b-9b2d-b7b674e314f6"))


## 9. OCR: Qwen Omni

`POST /llama/chatMulti`  
Модель: `Qwen/Qwen2.5-Omni-7B`. Картинка уходит multipart-полем `files`.


In [ ]:
def ocr_qwen(image_path: str, prompt: str = "Извлеки текст с картинки", token: str = TOKEN):
    request_json = {
        "chat": {
            "model": "Qwen/Qwen2.5-Omni-7B",
            "contents": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {}},
            ],
        }
    }
    ext = os.path.splitext(image_path)[1].lower()
    image_mime = "image/jpeg" if ext in (".jpg", ".jpeg") else "image/png"
    with open(image_path, "rb") as f:
        r = requests.post(
            f"{BASE}/llama/chatMulti",
            headers=headers(token, json_content=False),
            files={
                "files": (os.path.basename(image_path), f, image_mime),
                "request": (None, json.dumps(request_json), "application/json"),
            },
            verify=False,
            timeout=TIMEOUT,
        )
    if r.status_code != 200:
        return f"Error {r.status_code}: {r.text}"
    return r.json()


IMAGE_PATH = "/Users/anastasiamarkelova/Desktop/ex.png"
if os.path.exists(IMAGE_PATH):
    print(extract_text(ocr_qwen(IMAGE_PATH)))
else:
    print("Файл картинки не найден, ячейку можно запустить после подстановки IMAGE_PATH")


## 10. OCR: Llama 3.2 Vision

`POST /llama3_2/chat`  
Модель: `meta-llama/Llama-3.2-90B-Vision-Instruct`. Картинка кодируется в `data:image/png;base64,...`.


In [ ]:
def image_to_png_data_url(image_path: str) -> str:
    import base64

    from PIL import Image

    img = Image.open(image_path).convert("RGB")
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"


def ocr_llama32(
    image_path: str,
    prompt: str = "Извлеки текст с картинки.\nВерни только распознанный текст без комментариев.",
    token: str = TOKEN,
):
    image_path = Path(image_path)
    request_json = {
        "chat": {
            "model": "meta-llama/Llama-3.2-90B-Vision-Instruct",
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                    "contents": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": image_to_png_data_url(str(image_path)),
                                "detail": "auto",
                            },
                            "isUrl": True,
                            "fileName": image_path.with_suffix(".png").name,
                        },
                    ],
                }
            ],
            "temperature": 0.1,
            "top_p": 1,
            "top_k": 1000,
            "max_tokens": 2048,
            "n": 1,
        }
    }
    r = requests.post(
        f"{BASE}/llama3_2/chat",
        headers=headers(token, json_content=False),
        files={
            "request": (
                None,
                json.dumps(request_json, ensure_ascii=False),
                "application/json",
            )
        },
        verify=False,
        timeout=TIMEOUT,
    )
    if r.status_code != 200:
        return {"error": True, "status_code": r.status_code, "text": r.text}
    return r.json()


if os.path.exists(IMAGE_PATH):
    print(extract_text(ocr_llama32(IMAGE_PATH)))
else:
    print("Файл картинки не найден, ячейку можно запустить после подстановки IMAGE_PATH")
